In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr
import cartopy.crs as ccrs

In [2]:
x_data = xr.open_dataset("../../data/CanESM_1850-2100_tas.nc", engine="netcdf4")
y_data_rsut = xr.open_dataset("../../data/CanESM_1850-2100_rsutcre.nc", engine="netcdf4")
y_data_rlut = xr.open_dataset("../../data/CanESM_1850-2100_rlutcre.nc", engine="netcdf4")

print("Input dataset (SST): CanESM_1850-2100_tas.nc")
print("Label dataset (CRE): CanESM_1850-2100_rsutcre.nc")
print("Label dataset (CRE): CanESM_1850-2100_rlutcre.nc")

Input dataset (SST): CanESM_1850-2100_tas.nc
Label dataset (CRE): CanESM_1850-2100_rsutcre.nc
Label dataset (CRE): CanESM_1850-2100_rlutcre.nc


In [3]:
print("Input dataset (x_data / SST):")
print(x_data)
print("\nLabel dataset (y_data_rsut / RSUT):")
print(y_data_rsut)
print("\nLabel dataset (y_data_rlut / RLUT):")
print(y_data_rlut)

def select_var(ds, preferred_names):
    for name in preferred_names:
        if name in ds.data_vars:
            return name
    non_bnds = [name for name in ds.data_vars if not name.endswith("_bnds")]
    if not non_bnds:
        raise ValueError("No usable data variable found in dataset")
    return non_bnds[0]

x_var = select_var(x_data, ["tas"])
y_var_rsut = select_var(y_data_rsut, ["cre"])
y_var_rlut = select_var(y_data_rlut, ["cre"])


x_da = x_data[x_var]
y_da_rsut = y_data_rsut[y_var_rsut]
y_da_rlut = y_data_rlut[y_var_rlut]

print("\nSelected input variable:", x_var)
print(x_da)
print("\nSelected label variable (RSUT):", y_var_rsut)
print(y_da_rsut)
print("\nSelected label variable (RLUT):", y_var_rlut)
print(y_da_rlut)

Input dataset (x_data / SST):
<xarray.Dataset> Size: 2GB
Dimensions:  (member: 25, time: 3012, lat: 64, lon: 128)
Coordinates:
  * member   (member) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
    height   float64 8B ...
Data variables:
    tas      (member, time, lat, lon) float32 2GB ...

Label dataset (y_data_rsut / RSUT):
<xarray.Dataset> Size: 99MB
Dimensions:  (time: 3012, lat: 64, lon: 128)
Coordinates:
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data variables:
    cre      (time, lat, lon) float32 99MB ...

Label dataset (y_data_rlut / RLUT):

In [4]:
print("x_da dimensions:", x_da.dims)
print("x_da coordinates:", list(x_da.coords))
print("x_da attributes keys:", list(x_da.attrs.keys()))

print("\ny_da dimensions:", y_da_rsut.dims)
print("y_da coordinates:", list(y_da_rsut.coords))
print("y_da attributes keys:", list(y_da_rsut.attrs.keys()))

print("\ny_da dimensions:", y_da_rlut.dims)
print("y_da coordinates:", list(y_da_rlut.coords))
print("y_da attributes keys:", list(y_da_rlut.attrs.keys()))

x_da dimensions: ('member', 'time', 'lat', 'lon')
x_da coordinates: ['time', 'lat', 'lon', 'height', 'member']
x_da attributes keys: ['standard_name', 'long_name', 'comment', 'units', 'original_name', 'history', 'cell_methods', 'cell_measures']

y_da dimensions: ('time', 'lat', 'lon')
y_da coordinates: ['time', 'lat', 'lon']
y_da attributes keys: ['units', 'cell_methods', 'cell_measures', 'history', 'long_name', 'description']

y_da dimensions: ('time', 'lat', 'lon')
y_da coordinates: ['time', 'lat', 'lon']
y_da attributes keys: ['units', 'cell_methods', 'cell_measures', 'long_name', 'description']


In [ ]:
x_da, y_da_rsut, y_da_rlut = xr.align(x_da, y_da_rsut, y_da_rlut, join="inner")

shared_dims = [d for d in x_da.dims if d in y_da_rsut.dims and d in y_da_rlut.dims]
if not shared_dims:
    raise ValueError("No shared dimension found across input and both label datasets after alignment.")

preferred_split_dims = ["time", "year", "date", "sample", "index"]
split_dim = next((d for d in preferred_split_dims if d in shared_dims), None)

if split_dim is None:
    non_spatial_shared = [
        d for d in shared_dims
        if d.lower() not in {"lat", "latitude", "lon", "longitude", "bnds", "bounds", "nbnd", "nv"}
    ]
    if not non_spatial_shared:
        raise ValueError("Could not determine a valid non-spatial dimension to split on.")
    split_dim = non_spatial_shared[0]

n_samples = x_da.sizes[split_dim]
if n_samples < 3:
    raise ValueError(f"Need at least 3 samples along '{split_dim}' to create train/val/test splits.")

train_end = int(0.60 * n_samples)
val_end = int(0.80 * n_samples)

if train_end == 0 or val_end <= train_end or val_end >= n_samples:
    raise ValueError(
        f"Invalid split sizes for n={n_samples} along '{split_dim}'. "
        "Check data length or adjust split ratios."
    )

idx = np.arange(n_samples)
train_idx = idx[:train_end]
val_idx = idx[train_end:val_end]
test_idx = idx[val_end:]

X_train = x_da.isel({split_dim: train_idx})
X_val = x_da.isel({split_dim: val_idx})
X_test = x_da.isel({split_dim: test_idx})

y_train_rsut = y_da_rsut.isel({split_dim: train_idx})
y_val_rsut = y_da_rsut.isel({split_dim: val_idx})
y_test_rsut = y_da_rsut.isel({split_dim: test_idx})

y_train_rlut = y_da_rlut.isel({split_dim: train_idx})
y_val_rlut = y_da_rlut.isel({split_dim: val_idx})
y_test_rlut = y_da_rlut.isel({split_dim: test_idx})

print(f"Split dimension: {split_dim}")
print("Sequential split sizes (train/val/test):", len(train_idx), len(val_idx), len(test_idx))

if len(train_idx) > 0 and len(test_idx) > 0:
    print(
        f"Index ranges -> train: [{train_idx[0]}, {train_idx[-1]}], "
        f"val: [{val_idx[0]}, {val_idx[-1]}], "
        f"test: [{test_idx[0]}, {test_idx[-1]}]"
    )

print(f"X (SST) {split_dim} sizes:", X_train.sizes[split_dim], X_val.sizes[split_dim], X_test.sizes[split_dim])
print(f"y (RSUT) {split_dim} sizes:", y_train_rsut.sizes[split_dim], y_val_rsut.sizes[split_dim], y_test_rsut.sizes[split_dim])
print(f"y (RLUT) {split_dim} sizes:", y_train_rlut.sizes[split_dim], y_val_rlut.sizes[split_dim], y_test_rlut.sizes[split_dim])

ValueError: Expected a 'member' dimension in input and both label datasets.